In [2]:
import sys
sys.path.append('/rscratch/rishabhtiwari/QuantSpec/')

In [3]:
def expected_length(acceptance_rate, speculation_length):
    return (1-acceptance_rate**(speculation_length+1))/(1-acceptance_rate)

In [4]:
def expected_speedup(acceptance_rate, draft_speedup, speculation_length):
    return expected_length(acceptance_rate, speculation_length)/(speculation_length/draft_speedup+1)

In [10]:
expected_speedup(0.65, 100, 6)

2.5632822670990567

In [ ]:
expected_speedup(0.95, 3, 6)

2.41330163125

In [5]:
expected_length(0.9, 5)

4.68559

In [15]:
1/0.4049

2.469745616201531

In [ ]:
# 32, 64, 128
# 2, 3, 4

In [5]:
import torch
def simquant( inp, bits, q_channel=-1):
    maxval = torch.max(inp, dim=q_channel).values
    minval = torch.min(inp, dim=q_channel).values
    # print('inp', inp.shape)

    rangeval = (maxval - minval)
    print('rangeval', rangeval)

    qx = (2**bits - 1) / rangeval
    print('qx', qx)
    offset = minval * qx
    offset = offset.unsqueeze(q_channel)
    qx = qx.unsqueeze(q_channel)
    print('offset', offset)
    qinp = torch.round(qx * inp - offset)
    qinp = torch.clip(qinp, min=0, max=2**bits - 1)
    qinp_out = (qinp + offset) / qx
    qinp_out = torch.nan_to_num(qinp_out, nan=0.0, posinf=0.0, neginf=0.0)
    return qinp_out

In [110]:
0, 1, 2, 3

minval = 0
maxval = 3

rangeval = (maxval - minval) / 2
rangeval = 1.5


qx = 3/1.5

In [111]:
4096/16

256.0

In [6]:
a = torch.arange(-3, 3, dtype=torch.float16).unsqueeze(0).unsqueeze(2)

In [7]:
b = simquant(a, 2, -2)
b


rangeval tensor([[5.]], dtype=torch.float16)
qx tensor([[0.5996]], dtype=torch.float16)
offset tensor([[[-1.7988]]], dtype=torch.float16)


tensor([[[-3.0000],
         [-1.3320],
         [-1.3320],
         [ 0.3354],
         [ 0.3354],
         [ 2.0039]]], dtype=torch.float16)

In [9]:
a

tensor([[[-3.],
         [-2.],
         [-1.],
         [ 0.],
         [ 1.],
         [ 2.]]], dtype=torch.float16)

In [45]:
a = torch.randn(12, 32, 256, dtype=torch.float16) # h, s, d

In [46]:
a[0,0,0]

tensor(0.8750, dtype=torch.float16)

qx torch.Size([12, 1, 256])
offset torch.Size([12, 1, 256])


tensor(-0.1941, dtype=torch.float16)

In [19]:
a[0, 0, 0]

tensor(0.1434, dtype=torch.float16)

In [20]:
b[0, 0, 0]

tensor(-0.1395, dtype=torch.float16)

In [3]:
ls /rscratch/rishabhtiwari/KVQuant/quant/

dbrx/                     kvquant/           pyproject.toml
dbrx_simquant.py          kvquant.egg-info/  quantizers.pickle
eval_passkey_simquant.py  llama_simquant.py  README.md


In [3]:
import pickle
with open('/rscratch/rishabhtiwari/QuantSpec/quant_kivi/kvquant/quantizers.pickle', 'rb') as f:
    quantizers = pickle.load(f)

In [6]:
quantizers.keys()

dict_keys(['model.layers.0.self_attn.k_proj', 'model.layers.0.self_attn.v_proj', 'model.layers.1.self_attn.k_proj', 'model.layers.1.self_attn.v_proj', 'model.layers.2.self_attn.k_proj', 'model.layers.2.self_attn.v_proj', 'model.layers.3.self_attn.k_proj', 'model.layers.3.self_attn.v_proj', 'model.layers.4.self_attn.k_proj', 'model.layers.4.self_attn.v_proj', 'model.layers.5.self_attn.k_proj', 'model.layers.5.self_attn.v_proj', 'model.layers.6.self_attn.k_proj', 'model.layers.6.self_attn.v_proj', 'model.layers.7.self_attn.k_proj', 'model.layers.7.self_attn.v_proj', 'model.layers.8.self_attn.k_proj', 'model.layers.8.self_attn.v_proj', 'model.layers.9.self_attn.k_proj', 'model.layers.9.self_attn.v_proj', 'model.layers.10.self_attn.k_proj', 'model.layers.10.self_attn.v_proj', 'model.layers.11.self_attn.k_proj', 'model.layers.11.self_attn.v_proj', 'model.layers.12.self_attn.k_proj', 'model.layers.12.self_attn.v_proj', 'model.layers.13.self_attn.k_proj', 'model.layers.13.self_attn.v_proj', '

In [12]:
4096/8

512.0

In [11]:
quantizers['model.layers.0.self_attn.v_proj']

(tensor([[0.5713],
         [0.5425],
         [0.4170],
         ...,
         [0.4170],
         [0.5747],
         [0.3958]], dtype=torch.float64),
 tensor([[-0.4690],
         [-0.4343],
         [-0.4146],
         ...,
         [-0.4146],
         [-0.3652],
         [-0.3496]], dtype=torch.float64),
 [array([[-1.1584520e-01],
         [ 6.2970442e-01],
         [-4.6014488e-01],
         [ 8.0055743e-02],
         [-7.2442830e-01],
         [ 3.0609959e-01],
         [-3.5319477e-04],
         [-2.6002699e-01],
         [-9.7549957e-01],
         [-6.0416758e-02],
         [ 1.7456281e-01],
         [ 8.2220018e-01],
         [-1.8253192e-01],
         [-5.5151808e-01],
         [-3.4868342e-01],
         [ 4.3645167e-01]], dtype=float32)])

In [7]:
for i in quantizers.keys():
    print(i)


model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.5.self_attn.k_proj
model.layers.5.self_attn.v_proj
model.layers.6.self_attn.k_proj
model.layers.6.self_attn.v_proj
model.layers.7.self_attn.k_proj
model.layers.7.self_attn.v_proj
model.layers.8.self_attn.k_proj
model.layers.8.self_attn.v_proj
model.layers.9.self_attn.k_proj
model.layers.9.self_attn.v_proj
model.layers.10.self_attn.k_proj
model.layers.10.self_attn.v_proj
model.layers.11.self_attn.k_proj
model.layers.11.self_attn.v_proj
model.layers.12.self_attn.k_proj
model.layers.12.self_attn.v_proj
model.layers.13.self_attn.k_proj
model.layers.13.self_attn.v_proj
model.layers.14.self_attn.k_proj
model.layers.14.self_attn.v_proj
model.layers.15.self_attn.k_pr

In [15]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float16)
b = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

In [14]:
import torch

In [17]:
a[0, 0] = b[0, 1]

a.dtype, b.dtype

(torch.float16, torch.float32)